In [ ]:
# Cell 1: (no-op — repo is public, no PAT needed)
print('Repo is public — skipping secrets setup')


In [ ]:
# Cell 2: Clone repo (public)
%cd /kaggle/working
!rm -rf EMA-SKD
!git clone https://github.com/almaas-izdihar/4167a324fe EMA-SKD
%cd EMA-SKD
!git checkout experiment/confidence-filter
!git log --oneline -5


In [ ]:
# Cell 3: Verify GPU
!nvidia-smi

In [ ]:
# Cell 3.5: Run Config
END_EPOCH       = 2    # 2 = smoke test, 200 = full run
BATCH_PER_GPU   = 128  # each GPU runs independently
WORKERS_PER_GPU = 4

print(f'GPUs=2 (parallel, independent)  batch_per_gpu={BATCH_PER_GPU}  workers={WORKERS_PER_GPU}  epochs={END_EPOCH}')


In [ ]:
# Cell 3.6: Pre-download CIFAR-100 + start GPU monitor
import torchvision, os, subprocess, time

os.makedirs('/kaggle/working/data', exist_ok=True)
# Toronto mirror unreliable — attach cifar-100-python Kaggle dataset instead
# torchvision.datasets.CIFAR100(root='/kaggle/working/data', train=True,  download=True)
# torchvision.datasets.CIFAR100(root='/kaggle/working/data', train=False, download=True)
print('CIFAR-100 ready (loaded from attached dataset).')

# Start background GPU sampler — runs across Cell 4 and Cell 5
_gpu_log = '/kaggle/working/gpu_log.csv'
_gpu_proc = subprocess.Popen(
    f'nvidia-smi --query-gpu=timestamp,index,utilization.gpu,memory.used '
    f'--format=csv,noheader,nounits -l 5 > {_gpu_log}',
    shell=True
)
print(f'GPU logger started (PID {_gpu_proc.pid}) → {_gpu_log}')

In [ ]:
# Cell 4: Run baseline (GPU 0) and EMA-SKD conf-gate (GPU 1) in parallel
import subprocess, os, glob, time, sys

bs  = str(BATCH_PER_GPU)
ep  = str(END_EPOCH)
wk  = str(WORKERS_PER_GPU)

cmd_base = (
    f'CUDA_VISIBLE_DEVICES=0 python main.py '
    f'--data_type cifar100 --data_path /kaggle/working/data '
    f'--classifier_type ResNet18 --batch_size {bs} '
    f'--end_epoch {ep} --workers {wk} --seed 2024 '
    f'--experiment_type s0_baseline '
    f'> /kaggle/working/stdout_baseline.txt 2>&1'
)
cmd_ema = (
    f'CUDA_VISIBLE_DEVICES=1 python main.py '
    f'--data_type cifar100 --data_path /kaggle/working/data '
    f'--classifier_type ResNet18 --batch_size {bs} '
    f'--end_epoch {ep} --workers {wk} --seed 2024 '
    f'--beta 0.5 --EHSKD '
    f'--confidence_gate --tau_max 0.7 --tau_min 0.1 '
    f'--experiment_type s1_emaskd_conf_gate '
    f'> /kaggle/working/stdout_ema.txt 2>&1'
)

p_base = subprocess.Popen(cmd_base, shell=True)
p_ema  = subprocess.Popen(cmd_ema,  shell=True)
print(f'Baseline PID={p_base.pid} on GPU 0  |  EMA-SKD+gate PID={p_ema.pid} on GPU 1')
sys.stdout.flush()

def last_val_line(pattern):
    logs = sorted(glob.glob(f'models/*{pattern}*/log/log.txt'))
    if not logs:
        return 'no log yet'
    lines = [l for l in open(logs[-1]) if '[val]' in l]
    return lines[-1].strip() if lines else 'no val yet'

POLL = 60
while p_base.poll() is None or p_ema.poll() is None:
    time.sleep(POLL)
    base_done = '✓' if p_base.poll() is not None else '…'
    ema_done  = '✓' if p_ema.poll()  is not None else '…'
    print(f'[{time.strftime("%H:%M:%S")}] base{base_done} {last_val_line("EHSKD_False")}')
    print(f'[{time.strftime("%H:%M:%S")}]  ema{ema_done} {last_val_line("s1_emaskd")}')
    print()
    sys.stdout.flush()

p_base.wait()
p_ema.wait()
print('Both runs complete.')
sys.stdout.flush()


In [ ]:
# Cell 5: (merged into Cell 4 — both runs now launch in parallel above)
print('Parallel training launched in Cell 4.')


In [ ]:
# Cell 6: Metrics — parse logs and compare baseline vs EMA-SKD
import glob, re
import pandas as pd

def parse_log(path):
    rows = []
    with open(path) as f:
        for line in f:
            if '[val]' not in line:
                continue
            def g(key):
                m = re.search(rf'\[{key} ([^\]]+)\]', line)
                return float(m.group(1)) if m else None
            ep = re.search(r'\[Epoch (\d+)\]', line)
            if not ep:
                continue
            rows.append({
                'epoch':    int(ep.group(1)),
                'top1':     g('val_top1_acc'),
                'top5':     g('val_top5_acc'),
                'val_loss': g('val_loss'),
                'ece':      g('ECE'),
                'aurc':     g('AURC'),
                'eaurc':    g('EAURC'),
            })
    return pd.DataFrame(rows).set_index('epoch')

def find_latest_log(keyword):
    matches = sorted(glob.glob(f'models/*{keyword}*/log/log.txt'))
    if not matches:
        raise FileNotFoundError(f'No log matching: {keyword}')
    return matches[-1]

# Auto-detect: baseline = no EHSKD, ema = EHSKD_True
all_logs = sorted(glob.glob('models/*/log/log.txt'))
base_logs = [p for p in all_logs if 'EHSKD_False' in p]
ema_logs  = [p for p in all_logs if 'EHSKD_True'  in p]

if not base_logs or not ema_logs:
    raise FileNotFoundError(f'Logs not found.\nBaseline: {base_logs}\nEMA: {ema_logs}')

baseline_log = base_logs[-1]
ema_log      = ema_logs[-1]

df_base = parse_log(baseline_log)
df_ema  = parse_log(ema_log)

print('Baseline log:', baseline_log)
print('EMA-SKD log :', ema_log)
print(f'Epochs — Baseline: {len(df_base)}  EMA-SKD: {len(df_ema)}')
print()

last_base = df_base.iloc[-1]
last_ema  = df_ema.iloc[-1]

summary = pd.DataFrame({
    'Metric':   ['Top-1 Acc (%)', 'Top-5 Acc (%)', 'ECE (↓)', 'AURC (↓)', 'EAURC (↓)'],
    'Baseline': [last_base.top1, last_base.top5, last_base.ece, last_base.aurc, last_base.eaurc],
    'EMA-SKD':  [last_ema.top1,  last_ema.top5,  last_ema.ece,  last_ema.aurc,  last_ema.eaurc],
})
summary['Δ'] = summary['EMA-SKD'] - summary['Baseline']
print(summary.to_string(index=False, float_format=lambda x: f'{x:.3f}'))
print()
print('Paper targets — Baseline: 75.55 ± 0.09  |  EMA-SKD: 79.19 ± 0.15')

In [ ]:
# Cell 7: Training curves — Top-1, Val Loss, ECE, AURC per epoch
import glob, re, matplotlib.pyplot as plt, matplotlib.ticker as ticker
import pandas as pd

def _parse_log(path):
    rows = []
    with open(path) as f:
        for line in f:
            if '[val]' not in line: continue
            def g(key):
                m = re.search(rf'\[{key} ([^\]]+)\]', line)
                return float(m.group(1)) if m else None
            ep = re.search(r'\[Epoch (\d+)\]', line)
            if not ep: continue
            rows.append({'epoch': int(ep.group(1)), 'top1': g('val_top1_acc'),
                         'val_loss': g('val_loss'), 'ece': g('ECE'), 'aurc': g('AURC')})
    return pd.DataFrame(rows).set_index('epoch')

all_logs  = sorted(glob.glob('models/*/log/log.txt'))
base_logs = [p for p in all_logs if 'EHSKD_False' in p]
ema_logs  = [p for p in all_logs if 'EHSKD_True'  in p]
if not base_logs or not ema_logs:
    raise FileNotFoundError(f'Logs not found. Baseline={base_logs} EMA={ema_logs}')
_df_base = _parse_log(base_logs[-1])
_df_ema  = _parse_log(ema_logs[-1])

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('EMA-SKD vs Baseline — CIFAR-100 / ResNet18', fontsize=13)
panels = [
    ('top1',     'Top-1 Accuracy (%)', False),
    ('val_loss', 'Val Loss',           True),
    ('ece',      'ECE (↓)',            True),
    ('aurc',     'AURC (↓)',           True),
]
for ax, (col, title, _) in zip(axes.flat, panels):
    ax.plot(_df_base.index, _df_base[col], label='Baseline', marker='o', linewidth=1.5)
    ax.plot(_df_ema.index,  _df_ema[col],  label='EMA-SKD',  marker='s', linewidth=1.5)
    ax.set_title(title); ax.set_xlabel('Epoch')
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/eval_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: /kaggle/working/eval_curves.png')

# Crossover detection
_merged = _df_ema[['top1']].rename(columns={'top1':'ema'}).join(
    _df_base[['top1']].rename(columns={'top1':'base'}), how='inner')
_cross = _merged[_merged['ema'] > _merged['base']]
if _cross.empty:
    print('EMA-SKD did not exceed baseline — need more epochs or check config')
else:
    ep = _cross.index[0]
    print(f'EMA-SKD first exceeds baseline at epoch {ep}'
          f' (EMA {_cross.loc[ep,"ema"]:.3f}% vs Base {_cross.loc[ep,"base"]:.3f}%)')

# Final-epoch delta
print()
for col, title, _ in panels:
    b = _df_base[col].iloc[-1]; e = _df_ema[col].iloc[-1]
    print(f'{title:20s}  Baseline={b:.3f}  EMA-SKD={e:.3f}  Δ={e-b:+.3f}')


In [ ]:
# Cell 8: Resource usage — GPU utilization & memory timeseries
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Stop GPU logger
try:
    _gpu_proc.terminate()
    print(f'GPU logger stopped (PID {_gpu_proc.pid})')
except Exception as e:
    print(f'GPU logger stop: {e}')

# Parse CSV
df_gpu = pd.read_csv(
    '/kaggle/working/gpu_log.csv',
    names=['timestamp', 'gpu', 'util_pct', 'mem_mib'],
    skipinitialspace=True
)
df_gpu['timestamp'] = pd.to_datetime(df_gpu['timestamp'], format='%Y/%m/%d %H:%M:%S.%f', errors='coerce')
df_gpu = df_gpu.dropna(subset=['timestamp'])
df_gpu['t'] = (df_gpu['timestamp'] - df_gpu['timestamp'].min()).dt.total_seconds()
df_gpu['gpu'] = df_gpu['gpu'].astype(int)

# Save full numeric log as tsv (timestamp + t_sec + gpu + util + mem)
df_gpu.to_csv('/kaggle/working/gpu_log_parsed.tsv', sep='\t', index=False, float_format='%.1f')

# Save summary stats
stats = df_gpu.groupby('gpu')[['util_pct', 'mem_mib']].describe().round(1)
summary_lines = [
    f'GPU logger samples : {len(df_gpu)}',
    f'Total duration     : {df_gpu["t"].max():.0f}s',
    '',
    str(stats),
]
summary_text = '\n'.join(summary_lines)
with open('/kaggle/working/gpu_stats.txt', 'w') as f:
    f.write(summary_text)
print(summary_text)

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
fig.suptitle('GPU Resource Usage During Training (T4 x2)', fontsize=13)

styles = {0: dict(color='steelblue',   linestyle='-',  linewidth=1.5),
          1: dict(color='darkorange',   linestyle='--', linewidth=1.5)}

for gid in sorted(df_gpu['gpu'].unique()):
    sub = df_gpu[df_gpu['gpu'] == gid].sort_values('t')
    ax1.plot(sub['t'], sub['util_pct'], label=f'GPU {gid}', **styles.get(gid, {}))
    ax2.plot(sub['t'], sub['mem_mib'],  label=f'GPU {gid}', **styles.get(gid, {}))

ax1.set_ylabel('Utilization (%)')
ax1.set_title('GPU Utilization over Time')
ax1.set_ylim(0, 105)
ax1.yaxis.set_major_locator(ticker.MultipleLocator(20))
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.set_ylabel('Memory Used (MiB)')
ax2.set_title('GPU Memory over Time')
ax2.set_xlabel('Time (s)')
ax2.yaxis.set_major_locator(ticker.MultipleLocator(2000))
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/resource_usage.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: resource_usage.png  gpu_log.csv  gpu_log_parsed.tsv  gpu_stats.txt')
